# Silver Layer — Clean & Conform SaaS Data
Dedupe accounts/users, derive tenure and engagement bands. NO churn-derived flags (avoids leakage).

In [ ]:
from pyspark.sql.functions import (
    col, when, lit, to_timestamp, to_date, datediff, current_date,
    row_number, current_timestamp
)
from pyspark.sql.window import Window

In [ ]:
# Clean accounts (dedupe, derive tenure_days). Keep is_churned; drop churn_date leakage downstream in FE.
df_acc = spark.read.format('delta').table('bronze_accounts')
w = Window.partitionBy('account_id').orderBy(col('ingestion_timestamp').desc())
df_acc = (
    df_acc.withColumn('_rn', row_number().over(w)).filter(col('_rn') == 1).drop('_rn')
    .withColumn('signup_date', to_date('signup_date'))
    .withColumn('mrr_usd', col('mrr_usd').cast('double'))
    .withColumn('seat_count', col('seat_count').cast('int'))
    .withColumn('health_score', col('health_score').cast('double'))
    .withColumn('is_churned', col('is_churned').cast('int'))
    .withColumn('tenure_days', datediff(current_date(), col('signup_date')))
    .withColumn('silver_timestamp', current_timestamp())
)
df_acc.write.mode('overwrite').format('delta').saveAsTable('silver_accounts')
print(f'silver_accounts: {spark.table("silver_accounts").count()} rows')

In [ ]:
# Clean users
df_usr = spark.read.format('delta').table('bronze_users')
w2 = Window.partitionBy('user_id').orderBy(col('ingestion_timestamp').desc())
df_usr = (
    df_usr.withColumn('_rn', row_number().over(w2)).filter(col('_rn') == 1).drop('_rn')
    .withColumn('is_active', col('is_active').cast('int'))
    .withColumn('logins_last_30_days', col('logins_last_30_days').cast('int'))
    .withColumn('last_login_date', to_date('last_login_date'))
    .withColumn('engagement_band',
        when(col('logins_last_30_days') >= 20, 'Power User')
        .when(col('logins_last_30_days') >= 10, 'Regular')
        .when(col('logins_last_30_days') >= 1, 'Occasional')
        .otherwise('Dormant'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_usr.write.mode('overwrite').format('delta').saveAsTable('silver_users')
print(f'silver_users: {spark.table("silver_users").count()} rows')

In [ ]:
# Clean events
df_ev = spark.read.format('delta').table('bronze_events')
df_ev = (
    df_ev
    .withColumn('event_date', to_date('event_date'))
    .withColumn('duration_secs', col('duration_secs').cast('double'))
    .filter(col('event_date').isNotNull())
    .withColumn('silver_timestamp', current_timestamp())
)
df_ev.write.mode('overwrite').format('delta').saveAsTable('silver_events')
print(f'silver_events: {spark.table("silver_events").count()} rows')

In [ ]:
# Clean support tickets
df_tk = spark.read.format('delta').table('bronze_support_tickets')
df_tk = (
    df_tk
    .withColumn('created_at', to_timestamp('created_at'))
    .withColumn('resolution_hrs', col('resolution_hrs').cast('double'))
    .withColumn('sla_target_hrs', col('sla_target_hrs').cast('double'))
    .withColumn('is_sla_breached', col('is_sla_breached').cast('int'))
    .withColumn('csat_score', col('csat_score').cast('double'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_tk.write.mode('overwrite').format('delta').saveAsTable('silver_support_tickets')
print(f'silver_support_tickets: {spark.table("silver_support_tickets").count()} rows')